<a href="https://colab.research.google.com/github/WaitUps/CourseraExercises-AI-Agents-and-Agentic-AI-Course/blob/main/ProgrammaticPrompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Generative AI SDK: Gemini Integration Demo

**Adaptation Note:** This code has been adapted and refined from an earlier version originally designed for the OpenAI model, used as part of an exercise in the following Coursera course and specialization:

*   **Course:** [AI Agents and Agentic AI with Python & Generative AI](https://www.coursera.org/learn/ai-agents-python)
*   **Specialization:** [AI Agent Developer Specialization](https://www.coursera.org/specializations/ai-agents)


This notebook serves as a practical guide and demonstration for interacting with Google's Generative AI models (like Gemini) using the Python SDK within a Colab environment. It covers essential steps and best practices for integrating powerful AI capabilities into your applications.

# Section 1: Agentic AI Exercise - Gemini Adaptation for Core API Integration and Flexible Response Generation

This section focuses on the foundational aspects of integrating with Google's Generative AI models using the Python SDK. We'll cover API key management, model discovery, and implementing a versatile helper function for generating model responses.

### 1.1 Key Concepts

**Key features and concepts demonstrated:**

*   **API Key Management:** Securely configuring your Google API Key from Colab secrets.
*   **Model Discovery:** Listing available Generative AI models.
*   **Flexible Response Generation:** Implementing a `generate_response` helper function designed to:
    *   Interact seamlessly with various Gemini models.
    *   Handle system instructions (passing them directly or integrating into chat history based on model compatibility).
    *   Format messages correctly for the Gemini API.

The `generate_response` function is a component, built for adaptability and robust handling of model interactions, including detailed examples of how to interpret model outputs.

In [1]:
from google.colab import userdata # Imports the userdata module to access secrets stored in Google Colab
import google.generativeai as genai # Imports the Google Generative AI SDK

# Retrieve the API key from Colab's secrets manager.
# The key should be named 'GOOGLE_API_KEY' in the secrets tab.
api_key = userdata.get('GOOGLE_API_KEY')

# Configure the generative AI library with the retrieved API key.
# This makes the API key available for all subsequent `genai` calls in the notebook.
genai.configure(api_key=api_key)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [2]:
for m in genai.list_models():
  # Check if the model supports the 'generateContent' method.
  # This is crucial for models that are intended for text generation and conversational AI.
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [3]:
from typing import List, Dict # Used for type hinting to improve code readability and maintainability
import google.generativeai as genai # Imports the Google Generative AI SDK for interacting with Gemini models

def generate_response(messages: List[Dict]) -> str:
    """
    Calls the Gemini LLM to generate a response based on a list of messages.
    It handles system instructions separately if provided as the first message.
    """
    # API key handling:
    # The API key is configured globally via `genai.configure` in a previous cell.
    # This means the API key is set up once for all subsequent API calls with `genai`,
    # so there's no need to explicitly pass it or check environment variables within this function for `genai` models.

    system_instruction_content = ""

    # System Instruction Extraction:
    # The function checks if the first message in the list has the role 'system'.
    # If present, this message is treated as a system instruction to guide the model's behavior.
    if messages and messages[0]["role"] == "system":
        system_instruction_content = messages[0]["content"]
        # The remaining messages (from the second one onwards) form the actual chat history.
        chat_messages = messages[1:]
    else:
        # If no system instruction is provided, all messages are considered part of the chat history.
        chat_messages = messages

    # Message Formatting for Gemini API:
    # The GenerativeModel expects messages in a specific format for `generate_content`:
    # Each message should be a dictionary like `{'role': 'user'/'model', 'parts': [{'text': '...'}]}`.
    # This loop converts the simpler input message format `{'role': 'user'/'model', 'content': '...'}`
    # into the required structure by wrapping the content in a 'parts' list with a 'text' key.
    formatted_chat_messages = []
    for msg in chat_messages:
        # Assuming roles are 'user' or 'model' for standard chat interactions.
        formatted_chat_messages.append({'role': msg['role'], 'parts': [{'text': msg['content']}]})

    # Model Initialization:
    # A GenerativeModel instance is created, specifying the desired model ('gemini-flash-latest').
    # 'gemini-flash-latest' is chosen here as it generally offers a good balance of performance,
    # cost-effectiveness, and supports the `system_instruction` parameter directly, making it suitable
    # for many general-purpose tasks.
    # The extracted `system_instruction_content` is passed directly to the model during initialization.
    gemini_model = genai.GenerativeModel(
        'gemini-flash-latest',
        system_instruction=system_instruction_content # Pass the system instruction here, if any.
    )

    # Content Generation:
    # The `generate_content` method is called with the formatted chat messages.
    # `generation_config` is used to control various aspects of the generation process,
    # such as `max_output_tokens` (limiting the response length), `temperature` (creativity),
    # `top_p`, `top_k`, etc. Here, `max_output_tokens` is set to 4096.
    response = gemini_model.generate_content(
        formatted_chat_messages,
        generation_config=genai.GenerationConfig(max_output_tokens=4096)
    )
    # Return Value:
    # The generated text content from the model's response is extracted and returned.
    return response.text


In [4]:
messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)

In functional programming, we treat data as immutable and transformations as pure. When swapping keys and values, the primary concern is whether the transformation is **bijective** (one-to-one). If multiple keys share the same value, a simple swap will result in data loss.

Here are the two ways to handle this, ranging from a simple transformation to a more robust functional grouping.

### 1. The Simple Swap (Assuming unique values)
In Python, the most idiomatic functional approach is a **dictionary comprehension**. This is essentially syntactic sugar for a `map` operation over the dictionary's items.

```python
from typing import Dict, TypeVar

K = TypeVar('K')
V = TypeVar('V')

def swap_dict(d: Dict[K, V]) -> Dict[V, K]:
    """
    Swaps keys and values. 
    Note: If duplicate values exist, later keys will overwrite earlier ones.
    """
    return {v: k for k, v in d.items()}
```

### 2. The Robust Functional Swap (Handling Collisions)
If you want to ensure no data is lost when mu

## Sample of Generated Response (Agentic AI Exercise: Section 1):

*Note: This is a sample response generated by the AI model. Each execution of the `generate_response` function may produce a different output.*

<br>

---
<br>

In functional programming, we treat data transformations as a series of applications rather than a sequence of imperative steps.

When swapping keys and values, we must consider a critical edge case: **collisions**. Since dictionary keys must be unique but values do not, a simple swap might result in data loss if two keys share the same value.

Below are three ways to approach this, ranging from the "Pythonic" functional style to a more robust "grouping" approach.

### 1. The Declarative Approach (Dictionary Comprehension)
This is the most common functional pattern in Python. It is declarative, as it describes *what* the result should look like rather than *how* to loop through it.

```python
def swap_dict(d: dict) -> dict:
    return {v: k for k, v in d.items()}
```
*Note: If duplicate values exist, the last key encountered will overwrite previous ones.*

---

### 2. The Higher-Order Function Approach
If you prefer using point-free style or composition, you can use `map` and `reversed`. In this context, `reversed` acts on the key-value tuple.

```python
def swap_functional(d: dict) -> dict:
    # d.items() yields (key, value)
    # reversed turns (key, value) into (value, key)
    return dict(map(reversed, d.items()))
```

---

### 3. The Robust Approach (Handling Collisions)
A truly functional approach often avoids data loss. If multiple keys map to the same value, we should "fold" (reduce) the dictionary into a new one where values map to a *list* of keys.

We can achieve this using `functools.reduce`, which is the standard functional "fold" operation.

```python
from functools import reduce

def swap_and_group(d: dict) -> dict:
    def accumulate(acc, item):
        key, value = item
        # Return a new dict (or mutate local acc for performance)
        acc.setdefault(value, []).append(key)
        return acc

    return reduce(accumulate, d.items(), {})

# Example Usage:
data = {'a': 1, 'b': 2, 'c': 1}
print(swap_and_group

# Section 2: Building a Quasi-Agent for Python Function Generation

This section implements a 'quasi-agent' that demonstrates how to maintain context across multiple prompts with an LLM to build a Python function step-by-step. It asks the user for a function description, then uses sequential calls to the `generate_response` function (our LLM wrapper) to:

1.  Generate the basic Python function.
2.  Add comprehensive documentation to the function.
3.  Generate `unittest` test cases for the function.

Key improvements:
- The `develop_custom_function` leverages a growing message history to iteratively refine the generated Python function, documentation, and test cases.
- This allows for multi-turn interactions where subsequent LLM calls build upon previous responses, improving the coherence and completeness of the generated artifacts.
- Facilitates a more robust and 'agentic' development workflow by extending the agent's memory.

This exercise highlights managing information flow and handling LLM outputs, even when they might be inconsistent. We will leverage the `generate_response` function we defined earlier, which is configured to use `gemini-flash-latest` and handles system instructions and message formatting.

In [5]:
from typing import List, Dict
import sys
import os

def extract_code_block(response: str) -> str:
   """Extracts the first Python code block from a string response."""

   if '```' not in response:
      # If no code block markers, assume the entire response is code or not what we expect
      # For robustness, we might want to return an empty string or raise an error here
      return response.strip()

   # Split by ``` and take the content between the first two markers
   parts = response.split('```')
   if len(parts) < 3:
      # Not enough parts to contain a code block
      return response.strip()

   code_block = parts[1].strip()

   # Remove 'python' or 'py' if it's at the start of the code block
   if code_block.startswith("python"): # Covers 'python' and 'python\n'
      code_block = code_block[6:]
   elif code_block.startswith("py"): # Covers 'py' and 'py\n'
      code_block = code_block[2:]

   return code_block.strip()

In [6]:
def develop_custom_function():
   import io # Import the io module
   import sys # Import the sys module

   # Get user input for function description BEFORE redirecting stdout
   print("\nWhat kind of Python function would you like to create?")
   print("Example: 'A function that calculates the factorial of a number'")
   print("Your description: ", end='')
   function_description = input().strip()

   # Redirect stdout to a StringIO object to capture all subsequent print statements
   old_stdout = sys.stdout
   redirected_output = io.StringIO()
   sys.stdout = redirected_output

   try:
     # Initialize conversation with a system prompt. Our generate_response handles system messages.
     messages = [
        {"role": "system", "content": "You are a Python expert helping to develop a function. Always output code within a ```python code block```."}
     ]

     print("\n--- STEP 1: Generating Initial Function ---")
     # First prompt - Basic function
     user_prompt_1 = f"Write a basic Python function that {function_description}."
     messages.append({"role": "user", "content": user_prompt_1})

     llm_raw_response_1 = generate_response(messages)
     initial_function_code = extract_code_block(llm_raw_response_1)

     print("\n=== Initial Function (LLM Response) ===")
     print(initial_function_code)

     # Add LLM's response to conversation history.
     # We feed back the extracted code wrapped in markdown, as seen in the exercise example,
     # to provide clear context of what was 'generated' in the previous step.
     messages.append({"role": "model", "content": f"```python\n{initial_function_code}\n```"})

     print("\n--- STEP 2: Adding Documentation ---")
     # Second prompt - Add documentation
     user_prompt_2 = "Now, add comprehensive documentation to this function, including a description, parameter descriptions, return value description, example usage, and edge cases. Make sure the output is just the updated function in a ```python code block```."
     messages.append({"role": "user", "content": user_prompt_2})

     llm_raw_response_2 = generate_response(messages)
     documented_function_code = extract_code_block(llm_raw_response_2)

     print("\n=== Documented Function (LLM Response) ===")
     print(documented_function_code)

     # Add documented function response to conversation history
     messages.append({"role": "model", "content": f"```python\n{documented_function_code}\n```"})

     print("\n--- STEP 3: Adding Test Cases ---")
     # Third prompt - Add test cases
     user_prompt_3 = "Finally, add unittest test cases for this function. Tests should cover basic functionality, edge cases, and various input scenarios. Output the test cases code in a separate ```python code block```."
     messages.append({"role": "user", "content": user_prompt_3})

     llm_raw_response_3 = generate_response(messages)
     test_cases_code = extract_code_block(llm_raw_response_3)

     print("\n=== Test Cases (LLM Response) ===")
     print(test_cases_code)

     # Generate a filename from the function description
     filename_base = function_description.lower().replace('a function that ', '').replace('a function to ', '').strip()
     filename_base = ''.join(c for c in filename_base if c.isalnum() or c.isspace())
     filename_base = filename_base.replace(' ', '_')
     filename = f"{filename_base[:30] if len(filename_base) > 30 else filename_base}.py"

     # Ensure the filename is not empty or just an extension
     if not filename_base:
         filename = "generated_function.py"

     # Save final version to a Python file
     full_code = documented_function_code + '\n\n' + test_cases_code
     with open(filename, 'w') as f:
        f.write(full_code)

     print(f"\nFinal code has been saved to: {filename}")
     print("\n--- Quasi-Agent Development Process Complete ---")

     return documented_function_code, test_cases_code, filename, redirected_output.getvalue()
   finally:
     # Restore stdout
     sys.stdout = old_stdout

### Run the Quasi-Agent

Execute the cell below to start the quasi-agent. You will be prompted to enter a description for the Python function you want to create. The agent will then proceed through the steps of generating the function, adding documentation, and creating test cases, printing its progress along the way.

In [7]:
# Call the quasi-agent to develop a custom function
function_code, tests, filename, function_output = develop_custom_function()


What kind of Python function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: Converts a decimal number into its binary, octal, and hexadecimal 


In [8]:
print(f"\nFinal code has been saved to {filename}")
print(function_output)


Final code has been saved to converts_a_decimal_number_into.py

--- STEP 1: Generating Initial Function ---

=== Initial Function (LLM Response) ===
def convert_decimal(number):
    """
    Converts a decimal integer into binary, octal, and hexadecimal strings.
    """
    binary = bin(number)      # Returns string starting with '0b'
    octal = oct(number)        # Returns string starting with '0o'
    hexadecimal = hex(number)  # Returns string starting with '0x'

    return {
        "decimal": number,
        "binary": binary,
        "octal": octal,
        "hexadecimal": hexadecimal.upper() # upper() used for cleaner hex look
    }

# Example usage:
num = 255
results = convert_decimal(num)

print(f"Decimal: {results['decimal']}")
print(f"Binary: {results['binary']}")
print(f"Octal: {results['octal']}")
print(f"Hexadecimal: {results['hexadecimal']}")

--- STEP 2: Adding Documentation ---

=== Documented Function (LLM Response) ===
def convert_decimal(number: int) -> dict:
    """

## Sample Generated Output from `develop_custom_function()` (Quasi-Agent: Section 2):

--- STEP 1: Generating Initial Function ---

=== Initial Function (LLM Response) ===
def convert_decimal(number):
    """
    Converts a decimal number into binary, octal, and hexadecimal.
    
    Args:
        number (int): The decimal integer to convert.
        
    Returns:
        dict: A dictionary containing the converted strings.
    """
    return {
        "decimal": number,
        "binary": bin(number),      # Prefixed with '0b'
        "octal": oct(number),       # Prefixed with '0o'
        "hexadecimal": hex(number)  # Prefixed with '0x'
    }

# Example usage:
num = 255
results = convert_decimal(num)

print(f"Decimal: {results['decimal']}")
print(f"Binary: {results['binary']}")
print(f"Octal: {results['octal']}")
print(f"Hexadecimal: {results['hexadecimal']}")

# If you want the values without the prefixes (0b, 0o, 0x),
# you can use format string logic:
def convert_decimal_clean(number):
    return {
        "binary": format(number, 'b'),
        "octal": format(number, 'o'),
        "hexadecimal": format(number, 'x').upper()
    }

print("\nWithout prefixes:")
print(convert_decimal_clean(255))

--- STEP 2: Adding Documentation ---

=== Documented Function (LLM Response) ===
def convert_decimal(number):
    """
    Converts a decimal integer into its binary, octal, and hexadecimal representations.

    This function utilizes Python's built-in conversion functions to provide
    standard string representations of a base-10 integer in base-2, base-8,
    and base-16 formats.

    Args:
        number (int): The decimal integer to be converted.

    Returns:
        dict: A dictionary containing the converted values:
            - 'binary' (str): The binary representation (e.g., '0b1010').
            - 'octal' (str): The octal representation (e.g., '0o12').
            - 'hexadecimal' (str): The hexadecimal representation (e.g., '0xa').

    Example:
        >>> convert_decimal(255)
        {
            'binary': '0b11111111',
            'octal': '0o377',
            'hexadecimal': '0xff'
        }

    Edge Cases:
        - Zero (0): Returns '0b0', '0o0', and '0x0'.
        - Negative Integers: Python prefixes the negative sign before the
          base indicator (e.g., -10 becomes '-0b1010').
        - Non-Integer Inputs: Passing a float or a string will raise a
          TypeError, as bin(), oct(), and hex() require an integer.
    """
    return {
        "binary": bin(number),
        "octal": oct(number),
        "hexadecimal": hex(number)
    }

--- STEP 3: Adding Test Cases ---

=== Test Cases (LLM Response) ===
import unittest

def convert_decimal(number):
    """
    Converts a decimal integer into its binary, octal, and hexadecimal representations.
    """
    return {
        "binary": bin(number),
        "octal": oct(number),
        "hexadecimal": hex(number)
    }

class TestConvertDecimal(unittest.TestCase):
    
    def test_positive_integer(self):
        """Test conversion of a standard positive integer (10)."""
        result = convert_decimal(10)
        self.assertEqual(result['binary'], '0b1010')
        self.assertEqual(result['octal'], '0o12')
        self.assertEqual(result['hexadecimal'], '0xa')

    def test_large_integer(self):
        """Test conversion of a larger integer (255)."""
        result = convert_decimal(255)
        self.assertEqual(result['binary'], '0b11111111')
        self.assertEqual(result['octal'], '0o377')
        self.assertEqual(result['hexadecimal'], '0xff')

    def test_zero(self):
        """Test conversion of zero."""
        result = convert_decimal(0)
        self.assertEqual(result['binary'], '0b0')
        self.assertEqual(result['octal'], '0o0')
        self.assertEqual(result['hexadecimal'], '0x0')

    def test_negative_integer(self):
        """Test conversion of negative integers."""
        result = convert_decimal(-42)
        self.assertEqual(result['binary'], '-0b101010')
        self.assertEqual(result['octal'], '-0o52')
        self.assertEqual(result['hexadecimal'], '-0x2a')

    def test_invalid_float_input(self):
        """Test that passing a float raises a TypeError."""
        with self.assertRaises(TypeError):
            convert_decimal(10.5)

    def test_invalid_string_input(self):
        """Test that passing a string raises a TypeError."""
        with self.assertRaises(TypeError):
            convert_decimal("100")

    def test_return_type(self):
        """Ensure the function returns a dictionary with the correct keys."""
        result = convert_decimal(1)
        self.assertIsInstance(result, dict)
        self.assertIn('binary', result)
        self.assertIn('octal', result)
        self.assertIn('hexadecimal', result)

if __name__ == '__main__':
    unittest.main()

Final code has been saved to: converts_decimal_numbers_into_.py

--- Quasi-Agent Development Process Complete ---

